# **Bronze Layer - Synthetic Data Generation**

### Import configuration

In [0]:
%pip install faker

Looking in indexes: [REDACTED]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%restart_python

In [0]:
%run ../config/config.py

In [0]:
%run ../utils/helpers

In [0]:
import random
from datetime import datetime , timedelta

from faker import Faker

from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.types import *

### Initialize

In [0]:
random.seed(RANDOM_SEED)
apply_spark_optimizations(spark)
print("Bronze data generation started..")

Applying Spark optimizations...
⚠ Skipped: spark.sql.adaptive.enabled
⚠ Skipped: spark.sql.adaptive.coalescePartitions.enabled
⚠ Skipped: spark.sql.adaptive.skewJoin.enabled
✓ Applied: spark.sql.shuffle.partitions
⚠ Skipped: spark.sql.autoBroadcastJoinThreshold
✅ Spark optimization step completed.
Bronze data generation started..


In [0]:
fake = Faker("en_IN")
random.seed(RANDOM_SEED)
Faker.seed(RANDOM_SEED)
apply_spark_optimizations(spark)
print("Bronze data generation started..!")

Applying Spark optimizations...
⚠ Skipped: spark.sql.adaptive.enabled
⚠ Skipped: spark.sql.adaptive.coalescePartitions.enabled
⚠ Skipped: spark.sql.adaptive.skewJoin.enabled
✓ Applied: spark.sql.shuffle.partitions
⚠ Skipped: spark.sql.autoBroadcastJoinThreshold
✅ Spark optimization step completed.
Bronze data generation started..!


In [0]:
FIRST_NAMES_M = [...]
FIRST_NAMES_F = [...]
LAST_NAMES = [...]

In [0]:
num_patients = random.randint(* NUM_PATIENTS)

In [0]:
patients = []

In [0]:
for i in range(1, num_patients + 1):

    # Realistic hospital age distribution
    age_bucket = random.random()

    if age_bucket < 0.05:
        age = random.randint(1, 18)      # Pediatric
    elif age_bucket < 0.20:
        age = random.randint(19, 35)
    elif age_bucket < 0.55:
        age = random.randint(36, 60)
    elif age_bucket < 0.85:
        age = random.randint(61, 79)
    else:
        age = random.randint(80, 95)

    # Gender distribution
    gender = random.choices(
        ["F", "M", "Other"],
        weights=[0.49, 0.49, 0.02],
        k=1
    )[0]

    # Generate patient name
    # Generate patient name
    # Generate a realistic random Indian name
    name = fake.name()

    # Generate phone number
    contact = generate_phone()

    # Append patient record
    patients.append([
        f"P{str(i).zfill(5)}",
        name,
        age,
        gender,
        contact])
        

In [0]:
# for i in range(1, num_patients+1):

In [0]:
print(f"Generated {len(patients)} patients")
print("Sample Records:")
for patient in patients[:5]:
    print(patient)

Generated 205 patients
Sample Records:
['P00001', 'Aryan Maharaj', 61, 'M', '9265423511']
['P00002', 'Udant Dewan', 47, 'M', '9940781618']
['P00003', 'Gagan Sami', 56, 'M', '9593103413']
['P00004', 'Ayushman Chander', 92, 'F', '9525534192']
['P00005', 'Viraj Tiwari', 43, 'F', '9648350305']


In [0]:
# ==========================================================
# Create Patients Spark DataFrame
# ==========================================================

patient_schema = T.StructType([
    T.StructField("patient_id", T.StringType(), False),
    T.StructField("name", T.StringType(), True),
    T.StructField("age", T.IntegerType(), True),
    T.StructField("gender", T.StringType(), True),
    T.StructField("contact", T.StringType(), True)
])

patients_df = spark.createDataFrame(
    patients,
    schema=patient_schema
)

print("Patients Spark DataFrame created successfully.")
print(f"Rows: {patients_df.count()}")
patients_df.show(5, truncate=False)

Patients Spark DataFrame created successfully.
Rows: 205
+----------+----------------+---+------+----------+
|patient_id|name            |age|gender|contact   |
+----------+----------------+---+------+----------+
|P00001    |Aryan Maharaj   |61 |M     |9265423511|
|P00002    |Udant Dewan     |47 |M     |9940781618|
|P00003    |Gagan Sami      |56 |M     |9593103413|
|P00004    |Ayushman Chander|92 |F     |9525534192|
|P00005    |Viraj Tiwari    |43 |F     |9648350305|
+----------+----------------+---+------+----------+
only showing top 5 rows


In [0]:
# ==========================================================
# Write Patients Bronze Delta Table
# ==========================================================

print(f"Writing table: {BRONZE_PATIENTS}")

(
    patients_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(BRONZE_PATIENTS)
)

print(f"✅ {BRONZE_PATIENTS} written successfully.")

Writing table: dbacademy.smart_readmission.patients_bronze
✅ dbacademy.smart_readmission.patients_bronze written successfully.


### Generating Diagnosis Master data

In [0]:
# ==========================================================
# Generate Diagnosis Master Data
# ==========================================================

diagnoses = []

for diagnosis_id, icd_code, category in DIAGNOSIS_CATALOG:
    diagnoses.append([
        diagnosis_id,
        icd_code,
        category
    ])

print(f"Generated {len(diagnoses)} diagnoses")
print("Sample Records:")

for diagnosis in diagnoses[:5]:
    print(diagnosis)

Generated 12 diagnoses
Sample Records:
['D001', 'I21', 'Cardiovascular']
['D002', 'I50', 'Cardiovascular']
['D003', 'J18', 'Respiratory']
['D004', 'J44', 'Respiratory']
['D005', 'C34', 'Oncology']


### Converting Diagnosis to spark DataFrame

In [0]:
# ==========================================================
# Create Diagnosis Spark DataFrame
# ==========================================================

diagnosis_schema = T.StructType([
    T.StructField("diagnosis_id", T.StringType(), False),
    T.StructField("icd_code", T.StringType(), True),
    T.StructField("category", T.StringType(), True)
])

diagnoses_df = spark.createDataFrame(
    diagnoses,
    schema=diagnosis_schema
)

print("Diagnosis Spark DataFrame created successfully.")
print(f"Rows: {diagnoses_df.count()}")

diagnoses_df.show(12, truncate=False)

Diagnosis Spark DataFrame created successfully.
Rows: 12
+------------+--------+--------------+
|diagnosis_id|icd_code|category      |
+------------+--------+--------------+
|D001        |I21     |Cardiovascular|
|D002        |I50     |Cardiovascular|
|D003        |J18     |Respiratory   |
|D004        |J44     |Respiratory   |
|D005        |C34     |Oncology      |
|D006        |C50     |Oncology      |
|D007        |E11     |Endocrine     |
|D008        |N18     |Nephrology    |
|D009        |M17     |Orthopedics   |
|D010        |G40     |Neurology     |
|D011        |A41     |Infectious    |
|D012        |K35     |General       |
+------------+--------+--------------+



In [0]:
# ==========================================================
# Write Diagnoses Bronze Delta Table
# ==========================================================

print(f"Writing table: {BRONZE_DIAGNOSES}")

(
    diagnoses_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(BRONZE_DIAGNOSES)
)

print(f"✅ {BRONZE_DIAGNOSES} written successfully.")

Writing table: dbacademy.smart_readmission.diagnoses_bronze
✅ dbacademy.smart_readmission.diagnoses_bronze written successfully.


### Generating admission data

In [0]:
# ==========================================================
# Generate Admissions Data
# ==========================================================

admissions = []

num_admissions = random.randint(*NUM_ADMISSIONS)

diagnosis_ids = [d[0] for d in DIAGNOSIS_CATALOG]

patient_ids = [p[0] for p in patients]

for i in range(1, num_admissions + 1):

    # Select patient and diagnosis
    patient_id = random.choice(patient_ids)
    diagnosis_id = random.choice(diagnosis_ids)

    # Find diagnosis category
    diagnosis_category = next(
        d[2]
        for d in DIAGNOSIS_CATALOG
        if d[0] == diagnosis_id
    )

    # Select department based on diagnosis
    department_weights = CATEGORY_DEPARTMENT_MAP.get(
        diagnosis_category,
        {"General Medicine": 100}
    )

    department = weighted_choice(department_weights)

    # Admission date within the last year
    admission_date = datetime.now() - timedelta(
        days=random.randint(0, DATA_WINDOW_DAYS)
    )

    # Length of stay
    min_los, max_los = BASE_LOS_RANGES.get(
        department,
        (2, 6)
    )

    length_of_stay = random.randint(
        min_los,
        max_los
    )

    # Discharge date
    discharge_date = admission_date + timedelta(
        days=length_of_stay
    )

    # Physician
    physician = random.choice(
        PHYSICIANS.get(
            department,
            ["Dr. Kumar"]
        )
    )

    # Patient age
    patient_age = next(
        p[2]
        for p in patients
        if p[0] == patient_id
    )

    # ------------------------------------------------------
    # 30-Day Readmission Risk
    # ------------------------------------------------------

    readmission_probability = 0.10

    if patient_age >= 65:
        readmission_probability += 0.12
    elif patient_age >= 50:
        readmission_probability += 0.06

    if diagnosis_category in [
        "Cardiovascular",
        "Respiratory",
        "Oncology"
    ]:
        readmission_probability += 0.08

    if length_of_stay >= 7:
        readmission_probability += 0.06
    elif length_of_stay >= 5:
        readmission_probability += 0.03

    if department == "ICU":
        readmission_probability += 0.06

    readmitted_within_30_days = int(
        random.random() < min(readmission_probability, 0.60)
    )

    # ------------------------------------------------------
    # Append admission record
    # IMPORTANT: This is INSIDE the for loop
    # ------------------------------------------------------

    admissions.append([
        f"A{str(i).zfill(6)}",
        patient_id,
        diagnosis_id,
        admission_date.date(),
        discharge_date.date(),
        department,
        physician,
        length_of_stay,
        readmitted_within_30_days
    ])


# ==========================================================
# Validation
# ==========================================================

print(f"Generated {len(admissions)} admissions")

print("Sample Records:")

for admission in admissions[:5]:
    print(admission)

Generated 692 admissions
Sample Records:
['A000001', 'P00009', 'D002', datetime.date(2025, 11, 24), datetime.date(2025, 11, 28), 'General Medicine', 'Dr. Mehta', 4, 0]
['A000002', 'P00044', 'D007', datetime.date(2025, 9, 8), datetime.date(2025, 9, 10), 'General Medicine', 'Dr. Kumar', 2, 0]
['A000003', 'P00133', 'D005', datetime.date(2025, 12, 26), datetime.date(2026, 1, 1), 'Oncology', 'Dr. Roy', 6, 0]
['A000004', 'P00134', 'D006', datetime.date(2025, 11, 22), datetime.date(2025, 11, 28), 'Oncology', 'Dr. Das', 6, 0]
['A000005', 'P00012', 'D004', datetime.date(2026, 5, 30), datetime.date(2026, 6, 4), 'Pulmonology', 'Dr. Verma', 5, 0]


### Creating spark DataFrame

In [0]:
# ==========================================================
# Create Admissions Spark DataFrame
# ==========================================================

admission_schema = T.StructType([
    T.StructField("admission_id", T.StringType(), False),
    T.StructField("patient_id", T.StringType(), False),
    T.StructField("diagnosis_id", T.StringType(), False),
    T.StructField("admission_date", T.DateType(), True),
    T.StructField("discharge_date", T.DateType(), True),
    T.StructField("department", T.StringType(), True),
    T.StructField("physician", T.StringType(), True),
    T.StructField("length_of_stay", T.IntegerType(), True),
    T.StructField("readmitted_within_30_days", T.IntegerType(), True)
])

admissions_df = spark.createDataFrame(
    admissions,
    schema=admission_schema
)

print(f"Rows: {admissions_df.count()}")
admissions_df.show(10, truncate=False)

Rows: 692
+------------+----------+------------+--------------+--------------+----------------+---------+--------------+-------------------------+
|admission_id|patient_id|diagnosis_id|admission_date|discharge_date|department      |physician|length_of_stay|readmitted_within_30_days|
+------------+----------+------------+--------------+--------------+----------------+---------+--------------+-------------------------+
|A000001     |P00009    |D002        |2025-11-24    |2025-11-28    |General Medicine|Dr. Mehta|4             |0                        |
|A000002     |P00044    |D007        |2025-09-08    |2025-09-10    |General Medicine|Dr. Kumar|2             |0                        |
|A000003     |P00133    |D005        |2025-12-26    |2026-01-01    |Oncology        |Dr. Roy  |6             |0                        |
|A000004     |P00134    |D006        |2025-11-22    |2025-11-28    |Oncology        |Dr. Das  |6             |0                        |
|A000005     |P00012    |D004  

### Writing the bronze delta table

In [0]:
# ==========================================================
# Write Admissions Bronze Delta Table
# ==========================================================

print(f"Writing table: {BRONZE_ADMISSIONS}")

(
    admissions_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(BRONZE_ADMISSIONS)
)

print(f"✅ {BRONZE_ADMISSIONS} written successfully.")

Writing table: dbacademy.smart_readmission.admissions_bronze
✅ dbacademy.smart_readmission.admissions_bronze written successfully.
